# Nepali E-commerce Retrieval Dataset: EDA and Release Readiness

This notebook evaluates the integrity, composition, linguistic characteristics, retrieval difficulty, and practical usefulness of `nepali_ecommerce_ir_dataset.jsonl`. It also creates reproducible `train`, `test`, and `valid` splits, generates a Hugging Face dataset card, and includes a deliberately disabled upload cell.

The source JSONL is treated as immutable throughout the analysis.

In [4]:
from __future__ import annotations

import json
import math
import re
import unicodedata
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from datasets import Dataset, DatasetDict
from IPython.display import Markdown, display
from lets_plot import LetsPlot, aes, coord_flip, geom_bar, geom_boxplot, ggplot, ggsize, labs, theme_minimal
from sentence_transformers import SentenceTransformer
from sklearn.cluster import KMeans
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import normalize as sklearn_normalize

LetsPlot.setup_html()
pd.set_option("display.max_colwidth", 140)
pd.set_option("display.max_rows", 100)

MODEL_ID = "intfloat/multilingual-e5-small"
REPO_ID = "jangedoo/nepali-ecommerce-retrieval"
RANDOM_SEED = 42

data_candidates = [
    Path("nepali_ecommerce_ir_dataset.jsonl"),
    Path("nep_lms/datasets/nepali_ecommerce_qa/nepali_ecommerce_ir_dataset.jsonl"),
]
DATA_PATH = next((path.resolve() for path in data_candidates if path.exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError("Could not locate nepali_ecommerce_ir_dataset.jsonl")

CARD_PATH = DATA_PATH.parent / "README.md"
DATA_PATH

PosixPath('/home/jangedoo/Desktop/projects/nep-lms/nep_lms/datasets/nepali_ecommerce_qa/nepali_ecommerce_ir_dataset.jsonl')

## 1. Load and validate the source data

In [5]:
required_columns = [
    "row_id",
    "document",
    "negative1",
    "negative2",
    "negative3",
    "query",
    "product_category",
    "language_mode",
    "query_intent",
    "doc_type",
]

records = []
parse_errors = []
with DATA_PATH.open("r", encoding="utf-8") as handle:
    for line_number, line in enumerate(handle, start=1):
        try:
            records.append(json.loads(line))
        except json.JSONDecodeError as exc:
            parse_errors.append({"line": line_number, "error": str(exc)})

if parse_errors:
    raise ValueError(f"JSON parsing failed: {parse_errors[:5]}")

df = pd.DataFrame(records)
missing_columns = sorted(set(required_columns) - set(df.columns))
unexpected_columns = sorted(set(df.columns) - set(required_columns))
if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")

df = df[required_columns].copy()
display(df.head(3))
print(f"Rows: {len(df):,}")
print(f"Columns: {len(df.columns)}")
print(f"Unexpected columns: {unexpected_columns or 'none'}")

,row_id,document,negative1,negative2,negative3,query,product_category,language_mode,query_intent,doc_type
0,0,"The Samsung Galaxy S24 Ultra is now available in Nepal for NPR 1,59,999. It features a 200MP camera, S Pen support, and a powerful Snapd...","The Samsung Galaxy S24 FE is a more affordable option at NPR 75,999, with a 50MP camera and Exynos 2400 chipset. It offers 8GB RAM and 1...","The Samsung Galaxy Tab S9 Ultra is a high-end tablet priced at NPR 1,49,999. It features a 14.6-inch Dynamic AMOLED 2X display and comes...","The Samsung 65-inch 4K Smart TV is available for NPR 1,20,000. It supports HDR10+ and has built-in Alexa. This TV offers vibrant picture...",Samsung Galaxy S24 Ultra price in Nepal,Electronics,English only,direct_product_search,product_description
1,1,"I bought the 'Hatti Dhundo' black pashmina shawl from Himalayan Treasure for NPR 3,200 and it was incredibly soft and warm. However, my ...",The 'Hatti Dhundo' pashmina from Himalayan Treasure is made from pure cashmere wool and comes in various colors. It is machine washable ...,I ordered a traditional Nepali dhaka topi from Kathmandu Handicrafts for NPR 500. The fabric was high-quality and the embroidery was pre...,These running shoes from Nike are lightweight and have great arch support. I wear them for my daily jog at Ratna Park and they've lasted...,"कुन पश्मिना राम्रो छ, हिमालयन ट्रेजरको कि पश्मिना प्यालेसको?",Clothing & Fashion,Nepali (Devanagari),comparison_shopping,customer_review
2,2,"When comparing Fortune versus Sathi refined cooking oils, Fortune 1-liter bottle is priced at NPR 210 while Sathi's same size costs NPR ...","Fortune basmati rice 5kg is now available at NPR 850, which is a good deal compared to other premium brands like Kohinoor. Many customer...","Wai Wai noodles are a popular instant snack in Nepal, available in curry, chicken, and vegetable flavors. A single packet costs NPR 15, ...",Our store now offers free delivery on all orders above NPR 1000 within Kathmandu Valley. You can also earn reward points on every purcha...,sasto tel kun ramro fortuna ki sathi cooking oil price compare gara na,Groceries & Food,Romanized Nepali,price_inquiry,product_comparison


Rows: 1,990
Columns: 10
Unexpected columns: none


In [6]:
def normalize_query(text: str) -> str:
    text = unicodedata.normalize("NFKC", text).casefold()
    text = re.sub(r"[^\w\s]", " ", text)
    return re.sub(r"\s+", " ", text).strip()


text_columns = ["query", "document", "negative1", "negative2", "negative3"]
candidate_columns = ["document", "negative1", "negative2", "negative3"]

null_counts = df.isna().sum()
empty_counts = pd.Series(
    {
        column: int(df[column].astype(str).str.strip().eq("").sum())
        for column in df.columns
        if df[column].dtype == "object"
    },
    name="empty_strings",
)

row_ids = set(df["row_id"].astype(int))
missing_row_ids = sorted(set(range(min(row_ids), max(row_ids) + 1)) - row_ids)
duplicate_row_ids = int(df["row_id"].duplicated().sum())

exact_query_duplicate_rows = int(len(df) - df["query"].nunique())
exact_query_duplicate_groups = int((df.groupby("query").size() > 1).sum())
normalized_queries = df["query"].map(normalize_query)
normalized_query_duplicate_rows = int(len(df) - normalized_queries.nunique())

candidate_series = pd.concat([df[column] for column in candidate_columns], ignore_index=True)
unique_candidate_documents = int(candidate_series.nunique())
positive_negative_collision_mask = df.apply(
    lambda row: row["document"] in {row["negative1"], row["negative2"], row["negative3"]},
    axis=1,
)
positive_negative_collision_ids = df.loc[positive_negative_collision_mask, "row_id"].astype(int).tolist()

integrity_summary = pd.DataFrame(
    [
        ("JSON parse errors", len(parse_errors)),
        ("Missing required columns", len(missing_columns)),
        ("Null values", int(null_counts.sum())),
        ("Empty strings", int(empty_counts.sum())),
        ("Duplicate row_id values", duplicate_row_ids),
        ("Missing row_id values inside range", len(missing_row_ids)),
        ("Exact duplicate-query rows", exact_query_duplicate_rows),
        ("Exact duplicate-query groups", exact_query_duplicate_groups),
        ("Normalized duplicate-query rows", normalized_query_duplicate_rows),
        ("Positive/negative collision rows", len(positive_negative_collision_ids)),
        ("Unique candidate documents", unique_candidate_documents),
    ],
    columns=["check", "count"],
)
display(integrity_summary)
print("Missing row_id values:", missing_row_ids)
print("Positive/negative collision row_ids:", positive_negative_collision_ids)

,check,count
0,JSON parse errors,0
1,Missing required columns,0
2,Null values,0
3,Empty strings,0
4,Duplicate row_id values,0
5,Missing row_id values inside range,10
6,Exact duplicate-query rows,22
7,Exact duplicate-query groups,16
8,Normalized duplicate-query rows,24
9,Positive/negative collision rows,1


Missing row_id values: [361, 438, 455, 549, 748, 1223, 1254, 1514, 1593, 1983]
Positive/negative collision row_ids: [898]


In [7]:
duplicate_query_details = (
    df.assign(normalized_query=normalized_queries)
    .groupby("normalized_query", as_index=False)
    .agg(
        rows=("row_id", lambda values: list(map(int, values))),
        occurrences=("row_id", "size"),
        categories=("product_category", lambda values: sorted(set(values))),
        linked_positives=("document", "nunique"),
    )
    .query("occurrences > 1")
    .sort_values(["occurrences", "normalized_query"], ascending=[False, True])
)
display(duplicate_query_details)

if positive_negative_collision_ids:
    display(
        df.loc[
            df["row_id"].isin(positive_negative_collision_ids),
            ["row_id", "query", "document", "negative1", "negative2", "negative3"],
        ]
    )

,normalized_query,rows,occurrences,categories,linked_positives
404,fisher price laugh learn smart stages chair kati ma paaincha nepal ma,"[20, 440, 1280, 1700]",4,[Baby & Toys],4
1150,samsung galaxy s24 ultra price in nepal,"[0, 840, 1260, 1680]",4,[Electronics],4
888,pashmina shawl quality kasto cha reviews hernu hai,"[853, 1033, 1693]",3,[Clothing & Fashion],3
1155,samsung galaxy s24 ultra ra iphone 15 pro max nepal ma available cha ki chaina,"[576, 1416, 1836]",3,[Electronics],3
152,canon pixma g3270 printer price in nepal,"[70, 490]",2,[Office Supplies],2
406,fisher price laugh learn smart stages chair price nepal,"[140, 560]",2,[Baby & Toys],2
503,how to install pinlock on ls2 helmet,"[427, 1267]",2,[Automotive],2
564,is the decathlon forclaz trek 100 down jacket available in nepal,"[546, 1806]",2,[Sports & Outdoors],2
971,prestige induction cooktop price in nepal,"[700, 1540]",2,[Home & Kitchen],2
989,prestige omega deluxe induction cooktop price nepal kati ho,"[580, 1420]",2,[Home & Kitchen],2


,row_id,query,document,negative1,negative2,negative3
893,898,yo printer nepal ma shipping cost kati lagcha ra delivery time kati din lagcha?,The HP LaserJet Pro M404dn and Brother HL-L2350DW are both popular monochrome printers for office use. The HP model offers faster print ...,The HP LaserJet Pro M404dn and Brother HL-L2350DW are both popular monochrome printers for office use. The HP model offers faster print ...,The ErgoChair Pro and the Herman Miller Aeron are both ergonomic office chairs designed for long hours of sitting. The ErgoChair Pro has...,"Office paper is available in various sizes and weights, such as A4 70gsm and 80gsm. A ream of 500 sheets of JK Copier paper costs around..."


## 2. Composition, language, and text lengths

In [8]:
dimension_columns = ["product_category", "language_mode", "query_intent", "doc_type"]
distribution_tables = {}

for column in dimension_columns:
    counts = df[column].value_counts().rename_axis(column).reset_index(name="rows")
    counts["share_pct"] = (100 * counts["rows"] / len(df)).round(2)
    distribution_tables[column] = counts
    display(Markdown(f"### `{column}`"))
    display(counts)
    plot = (
        ggplot(counts, aes(x=column, y="rows", fill=column))
        + geom_bar(stat="identity", show_legend=False)
        + coord_flip()
        + theme_minimal()
        + ggsize(850, max(300, 32 * len(counts)))
        + labs(x="", y="Rows", title=f"Rows by {column}")
    )
    display(plot)

### `product_category`

,product_category,rows,share_pct
0,Electronics,167,8.39
1,Beauty & Health,167,8.39
2,Automotive,167,8.39
3,Clothing & Fashion,166,8.34
4,Groceries & Food,166,8.34
5,Books & Stationery,166,8.34
6,Home & Kitchen,166,8.34
7,Baby & Toys,166,8.34
8,Office Supplies,166,8.34
9,Sports & Outdoors,165,8.29


### `language_mode`

,language_mode,rows,share_pct
0,English only,285,14.32
1,Nepali (Devanagari),285,14.32
2,English-Nepali mixed,285,14.32
3,Romanized Nepali,284,14.27
4,Trilingual mixed,284,14.27
5,Romanized Nepali with English terms,284,14.27
6,English-Romanized Nepali,283,14.22


### `query_intent`

,query_intent,rows,share_pct
0,direct_product_search,200,10.05
1,price_inquiry,200,10.05
2,availability_inquiry,200,10.05
3,usage_and_installation,200,10.05
4,comparison_shopping,199,10.00
5,recommendation_request,199,10.00
6,warranty_and_returns,199,10.00
7,feature_and_specs,198,9.95
8,shipping_and_delivery,198,9.95
9,quality_and_reviews,197,9.90


### `doc_type`

,doc_type,rows,share_pct
0,product_description,285,14.32
1,customer_review,285,14.32
2,how_to_guide,285,14.32
3,product_comparison,284,14.27
4,shipping_policy,284,14.27
5,faq_answer,284,14.27
6,specification_sheet,283,14.22


In [9]:
def script_class(text: str) -> str:
    has_devanagari = bool(re.search(r"[\u0900-\u097F]", text))
    has_latin = bool(re.search(r"[A-Za-z]", text))
    if has_devanagari and has_latin:
        return "Devanagari + Latin"
    if has_devanagari:
        return "Devanagari only"
    if has_latin:
        return "Latin only"
    return "Other"


script_rows = []
for column in text_columns:
    counts = df[column].map(script_class).value_counts()
    for script, count in counts.items():
        script_rows.append({"field": column, "script": script, "rows": int(count)})
script_summary = pd.DataFrame(script_rows)
display(script_summary.pivot(index="field", columns="script", values="rows").fillna(0).astype(int))

query_script_by_label = pd.crosstab(df["language_mode"], df["query"].map(script_class))
display(query_script_by_label)

def contains_devanagari(text: str) -> bool:
    return any("\u0900" <= character <= "\u097f" for character in text)


positive_docs_with_devanagari = int(df["document"].map(contains_devanagari).sum())
negative_docs_with_devanagari = int(
    sum(df[column].map(contains_devanagari).sum() for column in ["negative1", "negative2", "negative3"])
)
print(f"Positive documents containing Devanagari: {positive_docs_with_devanagari:,} / {len(df):,}")
print(f"Negative documents containing Devanagari: {negative_docs_with_devanagari:,} / {3 * len(df):,}")

script,Devanagari + Latin,Devanagari only,Latin only
field,,,
document,0,0,1990
negative1,1,0,1989
negative2,0,0,1990
negative3,0,1,1989
query,210,250,1530


query,Devanagari + Latin,Devanagari only,Latin only
language_mode,,,
English only,0,0,285
English-Nepali mixed,138,0,147
English-Romanized Nepali,0,0,283
Nepali (Devanagari),35,250,0
Romanized Nepali,0,0,284
Romanized Nepali with English terms,0,0,284
Trilingual mixed,37,0,247


Positive documents containing Devanagari: 0 / 1,990
Negative documents containing Devanagari: 2 / 5,970


In [10]:
length_rows = []
for column in text_columns:
    words = df[column].str.split().str.len()
    characters = df[column].str.len()
    length_rows.append(
        {
            "field": column,
            "min_words": int(words.min()),
            "median_words": float(words.median()),
            "mean_words": round(float(words.mean()), 1),
            "max_words": int(words.max()),
            "median_characters": float(characters.median()),
            "mean_characters": round(float(characters.mean()), 1),
        }
    )
length_summary = pd.DataFrame(length_rows)
display(length_summary)

length_long = pd.concat(
    [pd.DataFrame({"field": column, "words": df[column].str.split().str.len()}) for column in text_columns],
    ignore_index=True,
)
display(
    ggplot(length_long, aes(x="field", y="words", fill="field"))
    + geom_boxplot(show_legend=False)
    + theme_minimal()
    + ggsize(850, 420)
    + labs(x="", y="Whitespace-token count", title="Text length by field")
)

,field,min_words,median_words,mean_words,max_words,median_characters,mean_characters
0,query,4,12.0,12.9,33,71.0,73.9
1,document,38,68.0,69.6,129,414.0,420.8
2,negative1,25,47.0,47.8,86,278.0,282.5
3,negative2,19,42.0,43.0,84,247.0,252.2
4,negative3,9,40.0,40.5,73,235.0,237.7


In [11]:
language_doc_type_table = pd.crosstab(df["language_mode"], df["doc_type"])
display(language_doc_type_table)

language_to_doc_types = df.groupby("language_mode")["doc_type"].nunique()
doc_type_to_languages = df.groupby("doc_type")["language_mode"].nunique()
language_doc_type_confounded = bool(
    language_to_doc_types.eq(1).all() and doc_type_to_languages.eq(1).all()
)
print("Language mode and document type are perfectly one-to-one confounded:", language_doc_type_confounded)

factor_tuple_count = int(df[dimension_columns].drop_duplicates().shape[0])
factor_tuple_theoretical = math.prod(df[column].nunique() for column in dimension_columns)
print(f"Observed four-factor combinations: {factor_tuple_count:,} / {factor_tuple_theoretical:,} possible")

doc_type,customer_review,faq_answer,how_to_guide,product_comparison,product_description,shipping_policy,specification_sheet
language_mode,,,,,,,
English only,0,0,0,0,285,0,0
English-Nepali mixed,0,0,285,0,0,0,0
English-Romanized Nepali,0,0,0,0,0,0,283
Nepali (Devanagari),285,0,0,0,0,0,0
Romanized Nepali,0,0,0,284,0,0,0
Romanized Nepali with English terms,0,284,0,0,0,0,0
Trilingual mixed,0,0,0,0,0,284,0


Language mode and document type are perfectly one-to-one confounded: True
Observed four-factor combinations: 420 / 5,880 possible


## 3. Multilingual E5 retrieval benchmark

In [12]:
def select_device() -> str:
    if not torch.cuda.is_available():
        return "cpu"
    try:
        free_memory = [torch.cuda.mem_get_info(index)[0] for index in range(torch.cuda.device_count())]
        return f"cuda:{int(np.argmax(free_memory))}"
    except Exception:
        return "cuda:0"


DEVICE = select_device()
torch.set_float32_matmul_precision("high")
print("Embedding device:", DEVICE)

model = SentenceTransformer(MODEL_ID, device=DEVICE)

candidate_documents = []
document_to_index = {}
for row in records:
    for column in candidate_columns:
        text = row[column]
        if text not in document_to_index:
            document_to_index[text] = len(candidate_documents)
            candidate_documents.append(text)

query_embeddings = model.encode(
    [f"query: {text}" for text in df["query"]],
    batch_size=128,
    normalize_embeddings=True,
    show_progress_bar=True,
)
document_embeddings = model.encode(
    [f"passage: {text}" for text in candidate_documents],
    batch_size=128,
    normalize_embeddings=True,
    show_progress_bar=True,
)
positive_document_indices = np.array([document_to_index[text] for text in df["document"]])
positive_embeddings = document_embeddings[positive_document_indices]

print("Query embedding shape:", query_embeddings.shape)
print("Candidate-document embedding shape:", document_embeddings.shape)

Embedding device: cuda:0


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Batches:   0%|          | 0/16 [00:00<?, ?it/s]

Batches:   0%|          | 0/63 [00:00<?, ?it/s]

Query embedding shape: (1990, 384)
Candidate-document embedding shape: (7959, 384)


In [13]:
local_ranks = []
local_margins = []
local_row_indices = []

for row_index, row in df.iterrows():
    candidate_texts = [row[column] for column in candidate_columns]
    if len(set(candidate_texts)) < 4:
        continue
    candidate_indices = [document_to_index[text] for text in candidate_texts]
    scores = query_embeddings[row_index] @ document_embeddings[candidate_indices].T
    ordering = np.argsort(-scores)
    rank = int(np.flatnonzero(ordering == 0)[0] + 1)
    local_ranks.append(rank)
    local_margins.append(float(scores[0] - np.max(scores[1:])))
    local_row_indices.append(row_index)

local_ranks = np.asarray(local_ranks)
local_margins = np.asarray(local_margins)
local_metrics = {
    "evaluated_rows": int(len(local_ranks)),
    "accuracy": float(np.mean(local_ranks == 1)),
    "mrr": float(np.mean(1 / local_ranks)),
    "mean_positive_margin": float(local_margins.mean()),
    "non_positive_margin_share": float(np.mean(local_margins <= 0)),
}
local_rank_distribution = pd.Series(local_ranks).value_counts().sort_index().rename_axis("rank").reset_index(name="rows")
display(pd.DataFrame([local_metrics]).round(4))
display(local_rank_distribution)

,evaluated_rows,accuracy,mrr,mean_positive_margin,non_positive_margin_share
0,1989,0.9175,0.9571,0.0373,0.0825


,rank,rows
0,1,1825
1,2,146
2,3,15
3,4,3


In [14]:
relevant_document_indices = defaultdict(set)
for row_index, row in df.iterrows():
    relevant_document_indices[normalize_query(row["query"])].add(positive_document_indices[row_index])

global_ranks = np.empty(len(df), dtype=np.int32)
batch_size = 128
for start in range(0, len(df), batch_size):
    stop = min(start + batch_size, len(df))
    scores = query_embeddings[start:stop] @ document_embeddings.T
    for offset, score_vector in enumerate(scores):
        row_index = start + offset
        relevant = list(relevant_document_indices[normalized_queries.iloc[row_index]])
        best_relevant_score = float(np.max(score_vector[relevant]))
        global_ranks[row_index] = 1 + int(np.sum(score_vector > best_relevant_score))


def retrieval_metrics(ranks: np.ndarray) -> dict[str, float]:
    ranks = np.asarray(ranks)
    return {
        "rows": int(len(ranks)),
        "Recall@1": float(np.mean(ranks <= 1)),
        "Recall@5": float(np.mean(ranks <= 5)),
        "Recall@10": float(np.mean(ranks <= 10)),
        "MRR@10": float(np.mean(np.where(ranks <= 10, 1 / ranks, 0))),
        "median_rank": float(np.median(ranks)),
    }


global_metrics = retrieval_metrics(global_ranks)
display(pd.DataFrame([global_metrics]).round(4))

,rows,Recall@1,Recall@5,Recall@10,MRR@10,median_rank
0,1990,0.2106,0.4719,0.5864,0.3222,6.0


In [15]:
def metrics_by(column: str) -> pd.DataFrame:
    rows = []
    for value, indices in df.groupby(column).groups.items():
        metrics = retrieval_metrics(global_ranks[np.asarray(list(indices))])
        rows.append({column: value, **metrics})
    return pd.DataFrame(rows).sort_values("Recall@10", ascending=False).reset_index(drop=True)


metrics_by_dimension = {column: metrics_by(column) for column in dimension_columns}
for column, table in metrics_by_dimension.items():
    display(Markdown(f"### Retrieval metrics by `{column}`"))
    display(table.round(4))

language_metric_plot = metrics_by_dimension["language_mode"][["language_mode", "Recall@1", "Recall@10"]].melt(
    id_vars="language_mode", var_name="metric", value_name="score"
)
display(
    ggplot(language_metric_plot, aes(x="language_mode", y="score", fill="metric"))
    + geom_bar(stat="identity", position="dodge")
    + coord_flip()
    + theme_minimal()
    + ggsize(900, 460)
    + labs(x="", y="Score", title="Full-corpus E5 retrieval by labeled language mode")
)

### Retrieval metrics by `product_category`

,product_category,rows,Recall@1,Recall@5,Recall@10,MRR@10,median_rank
0,Home & Kitchen,166,0.2349,0.5542,0.6687,0.3733,4.0
1,Beauty & Health,167,0.2994,0.5569,0.6647,0.4157,4.0
2,Sports & Outdoors,165,0.2303,0.4970,0.6424,0.3460,6.0
3,Office Supplies,166,0.2108,0.5000,0.6325,0.3320,5.5
4,Automotive,167,0.2575,0.5329,0.6287,0.3805,4.0
5,Electronics,167,0.1677,0.5090,0.6168,0.3053,5.0
6,Books & Stationery,166,0.3133,0.5301,0.6145,0.4090,4.0
7,Pet Supplies,164,0.1707,0.4939,0.6037,0.3033,6.0
8,Groceries & Food,166,0.2651,0.4759,0.5904,0.3606,6.0
9,Baby & Toys,166,0.1506,0.3614,0.4819,0.2463,12.0


### Retrieval metrics by `language_mode`

,language_mode,rows,Recall@1,Recall@5,Recall@10,MRR@10,median_rank
0,Romanized Nepali with English terms,284,0.3028,0.5845,0.7394,0.4275,4.0
1,English only,285,0.2632,0.5719,0.6807,0.3888,4.0
2,Romanized Nepali,284,0.2676,0.5563,0.6725,0.3875,4.5
3,English-Romanized Nepali,283,0.1661,0.4735,0.5830,0.2996,7.0
4,English-Nepali mixed,285,0.2316,0.4772,0.5754,0.3356,6.0
5,Trilingual mixed,284,0.1514,0.3908,0.5035,0.2493,10.0
6,Nepali (Devanagari),285,0.0912,0.2491,0.3509,0.1675,26.0


### Retrieval metrics by `query_intent`

,query_intent,rows,Recall@1,Recall@5,Recall@10,MRR@10,median_rank
0,comparison_shopping,199,0.3970,0.6432,0.7588,0.5023,2.0
1,availability_inquiry,200,0.2150,0.5600,0.6750,0.3600,4.0
2,direct_product_search,200,0.2500,0.5350,0.6500,0.3673,5.0
3,feature_and_specs,198,0.2525,0.5152,0.6465,0.3684,5.0
4,usage_and_installation,200,0.2050,0.5350,0.6450,0.3465,5.0
5,price_inquiry,200,0.1400,0.3750,0.5400,0.2501,9.0
6,warranty_and_returns,199,0.1457,0.4121,0.5176,0.2566,10.0
7,shipping_and_delivery,198,0.1919,0.4293,0.5152,0.2903,9.5
8,quality_and_reviews,197,0.1929,0.3807,0.4822,0.2774,12.0
9,recommendation_request,199,0.1156,0.3317,0.4322,0.2027,16.0


### Retrieval metrics by `doc_type`

,doc_type,rows,Recall@1,Recall@5,Recall@10,MRR@10,median_rank
0,faq_answer,284,0.3028,0.5845,0.7394,0.4275,4.0
1,product_description,285,0.2632,0.5719,0.6807,0.3888,4.0
2,product_comparison,284,0.2676,0.5563,0.6725,0.3875,4.5
3,specification_sheet,283,0.1661,0.4735,0.5830,0.2996,7.0
4,how_to_guide,285,0.2316,0.4772,0.5754,0.3356,6.0
5,shipping_policy,284,0.1514,0.3908,0.5035,0.2493,10.0
6,customer_review,285,0.0912,0.2491,0.3509,0.1675,26.0


In [16]:
failure_cases = df[["row_id", "query", "product_category", "language_mode", "query_intent", "doc_type"]].copy()
failure_cases["global_rank"] = global_ranks
failure_cases = failure_cases.sort_values("global_rank", ascending=False).head(15)
display(failure_cases)

,row_id,query,product_category,language_mode,query_intent,doc_type,global_rank
817,822,yoga mat ko price kati ho budget friendly ma?,Sports & Outdoors,English-Nepali mixed,price_inquiry,how_to_guide,5914
1501,1508,यो Chicco stroller Nepal ma kati din ma aaucha ra shipping cost kati lagcha?,Baby & Toys,English-Nepali mixed,shipping_and_delivery,how_to_guide,3810
302,302,यो चामलको मूल्य कति हो र किफायती छ?,Groceries & Food,Nepali (Devanagari),price_inquiry,customer_review,3683
387,388,यो modular sofa ko shipping cost kati hola Nepal ma? कस्तो delivery time cha?,Home & Kitchen,English-Nepali mixed,shipping_and_delivery,how_to_guide,3434
1849,1858,canon printer nepal ma shipping charge kati ho ra delivery time kati lagcha?,Office Supplies,English-Nepali mixed,shipping_and_delivery,how_to_guide,3388
1426,1433,yo skincare product genuine ho ki nai? kasaiko review cha? quality kasto cha?,Beauty & Health,Trilingual mixed,quality_and_reviews,shipping_policy,3302
1844,1853,yo skin care product ko quality kasto cha reviews haru kahan paaincha? genuine customer feedback hernu cha,Beauty & Health,Trilingual mixed,quality_and_reviews,shipping_policy,2750
1431,1438,यो ErgoChair को shipping cost Nepal ma kati lagcha? Delivery time kati ho?,Office Supplies,English-Nepali mixed,shipping_and_delivery,how_to_guide,2549
594,598,यो ergonomic chair ko shipping cost Nepal ma kati lagcha?,Office Supplies,English-Nepali mixed,shipping_and_delivery,how_to_guide,2493
1137,1142,यो मसला कति पर्छ?,Groceries & Food,Nepali (Devanagari),price_inquiry,customer_review,2435


## 4. E5 semantic-cluster split construction

In [17]:
# Equal-weight concatenation retains both query and positive-document semantics.
pair_embeddings = sklearn_normalize(np.concatenate([query_embeddings, positive_embeddings], axis=1))

row_cluster_label = np.full(len(df), -1, dtype=np.int16)
selected_cluster_by_category = {}
cluster_group_data = {}
cluster_summary_rows = []
test_indices = set()

for category in sorted(df["product_category"].unique()):
    category_indices = df.index[df["product_category"].eq(category)].tolist()
    query_groups = defaultdict(list)
    for row_index in category_indices:
        query_groups[normalized_queries.iloc[row_index]].append(row_index)

    group_keys = sorted(query_groups)
    group_embeddings = sklearn_normalize(
        np.stack([pair_embeddings[query_groups[key]].mean(axis=0) for key in group_keys])
    )
    kmeans = KMeans(n_clusters=5, n_init=20, random_state=RANDOM_SEED).fit(group_embeddings)
    cluster_centroids = sklearn_normalize(kmeans.cluster_centers_)
    centroid_similarities = cluster_centroids @ cluster_centroids.T

    for key, label in zip(group_keys, kmeans.labels_):
        for row_index in query_groups[key]:
            row_cluster_label[row_index] = int(label)

    cluster_sizes = np.array(
        [
            sum(len(query_groups[key]) for key, label in zip(group_keys, kmeans.labels_) if label == cluster_id)
            for cluster_id in range(5)
        ]
    )
    cluster_shares = cluster_sizes / len(category_indices)
    nearest_centroid_distances = np.array(
        [
            1 - np.max(np.delete(centroid_similarities[cluster_id], cluster_id))
            for cluster_id in range(5)
        ]
    )

    eligible = np.flatnonzero((cluster_shares >= 0.10) & (cluster_shares <= 0.30)).tolist()
    if eligible:
        selected_cluster = max(
            eligible,
            key=lambda cluster_id: (
                nearest_centroid_distances[cluster_id],
                -abs(cluster_shares[cluster_id] - 0.20),
                -cluster_id,
            ),
        )
    else:
        selected_cluster = min(
            range(5),
            key=lambda cluster_id: (
                abs(cluster_shares[cluster_id] - 0.20),
                -nearest_centroid_distances[cluster_id],
                cluster_id,
            ),
        )

    selected_cluster_by_category[category] = int(selected_cluster)
    selected_rows = {
        row_index for row_index in category_indices if row_cluster_label[row_index] == selected_cluster
    }
    test_indices.update(selected_rows)
    cluster_group_data[category] = (group_embeddings, kmeans.labels_, int(selected_cluster))

    for cluster_id in range(5):
        cluster_summary_rows.append(
            {
                "product_category": category,
                "cluster_id": cluster_id,
                "rows": int(cluster_sizes[cluster_id]),
                "share": float(cluster_shares[cluster_id]),
                "nearest_centroid_cosine_distance": float(nearest_centroid_distances[cluster_id]),
                "selected_for_test": cluster_id == selected_cluster,
            }
        )

cluster_summary = pd.DataFrame(cluster_summary_rows)
display(cluster_summary.round(4))
display(cluster_summary.query("selected_for_test").reset_index(drop=True).round(4))

,product_category,cluster_id,rows,share,nearest_centroid_cosine_distance,selected_for_test
0,Automotive,0,32,0.1916,0.0248,False
1,Automotive,1,30,0.1796,0.0403,True
2,Automotive,2,54,0.3234,0.0248,False
3,Automotive,3,23,0.1377,0.0403,False
4,Automotive,4,28,0.1677,0.0363,False
5,Baby & Toys,0,34,0.2048,0.0335,True
6,Baby & Toys,1,64,0.3855,0.0322,False
7,Baby & Toys,2,38,0.2289,0.0335,False
8,Baby & Toys,3,11,0.0663,0.0336,False
9,Baby & Toys,4,19,0.1145,0.0322,False


,product_category,cluster_id,rows,share,nearest_centroid_cosine_distance,selected_for_test
0,Automotive,1,30,0.1796,0.0403,True
1,Baby & Toys,0,34,0.2048,0.0335,True
2,Beauty & Health,1,19,0.1138,0.0423,True
3,Books & Stationery,4,34,0.2048,0.0448,True
4,Clothing & Fashion,0,48,0.2892,0.0518,True
5,Electronics,1,44,0.2635,0.0428,True
6,Groceries & Food,0,39,0.2349,0.0502,True
7,Home & Kitchen,2,37,0.2229,0.0336,True
8,Mobile & Accessories,4,36,0.2195,0.0508,True
9,Office Supplies,3,33,0.1988,0.0378,True


In [18]:
remaining_indices = sorted(set(df.index) - test_indices)
target_valid_size = round(0.10 * len(df))
stratification_columns = ["product_category", "language_mode", "query_intent", "doc_type"]


def marginal_distribution(indices, column):
    counts = df.loc[list(indices), column].value_counts()
    return {value: counts.get(value, 0) / len(indices) for value in df[column].unique()}


def total_variation(left, right):
    return 0.5 * sum(abs(left.get(key, 0) - right.get(key, 0)) for key in set(left) | set(right))


remaining_marginals = {
    column: marginal_distribution(remaining_indices, column) for column in stratification_columns
}
remaining_groups = np.array([normalized_queries.iloc[index] for index in remaining_indices])

best_candidate = None
for seed in range(200):
    splitter = GroupShuffleSplit(
        n_splits=1,
        test_size=target_valid_size / len(remaining_indices),
        random_state=seed,
    )
    _, valid_positions = next(
        splitter.split(np.arange(len(remaining_indices)), groups=remaining_groups)
    )
    candidate_valid = [remaining_indices[position] for position in valid_positions]
    size_error = abs(len(candidate_valid) - target_valid_size) / len(df)
    distribution_error = sum(
        total_variation(
            marginal_distribution(candidate_valid, column),
            remaining_marginals[column],
        )
        for column in stratification_columns
    )
    candidate = (size_error + distribution_error, abs(len(candidate_valid) - target_valid_size), seed, candidate_valid)
    if best_candidate is None or candidate[:3] < best_candidate[:3]:
        best_candidate = candidate

valid_selection_score, valid_size_error, valid_seed, candidate_valid = best_candidate
valid_indices = set(candidate_valid)
train_indices = set(remaining_indices) - valid_indices

split_membership = pd.Series(index=df.index, dtype="object")
split_membership.loc[sorted(train_indices)] = "train"
split_membership.loc[sorted(test_indices)] = "test"
split_membership.loc[sorted(valid_indices)] = "valid"

split_counts = split_membership.value_counts().reindex(["train", "test", "valid"])
display(split_counts.rename("rows").to_frame())
print("Validation candidate seed:", valid_seed)
print("Validation selection score:", round(valid_selection_score, 4))

,rows
train,1367
test,425
valid,198


Validation candidate seed: 90
Validation selection score: 0.1972


In [19]:
expected_split_counts = {"train": 1367, "test": 425, "valid": 198}
assert split_counts.to_dict() == expected_split_counts, (split_counts.to_dict(), expected_split_counts)
assert not split_membership.isna().any()

split_indices = {
    split: set(split_membership.index[split_membership.eq(split)])
    for split in ["train", "test", "valid"]
}

for column in stratification_columns:
    full_values = set(df[column])
    for split, indices in split_indices.items():
        assert set(df.loc[list(indices), column]) == full_values, f"{split} lacks values for {column}"

for left, right in [("train", "test"), ("train", "valid"), ("test", "valid")]:
    left_queries = set(normalized_queries.loc[list(split_indices[left])])
    right_queries = set(normalized_queries.loc[list(split_indices[right])])
    assert left_queries.isdisjoint(right_queries), f"Query leakage between {left} and {right}"

for row_index in test_indices:
    category = df.at[row_index, "product_category"]
    assert row_cluster_label[row_index] == selected_cluster_by_category[category]
for row_index in train_indices | valid_indices:
    category = df.at[row_index, "product_category"]
    assert row_cluster_label[row_index] != selected_cluster_by_category[category]

split_coverage_rows = []
for split, indices in split_indices.items():
    split_coverage_rows.append(
        {
            "split": split,
            "rows": len(indices),
            "product_categories": df.loc[list(indices), "product_category"].nunique(),
            "language_modes": df.loc[list(indices), "language_mode"].nunique(),
            "query_intents": df.loc[list(indices), "query_intent"].nunique(),
            "document_types": df.loc[list(indices), "doc_type"].nunique(),
        }
    )
split_coverage = pd.DataFrame(split_coverage_rows)
display(split_coverage)

,split,rows,product_categories,language_modes,query_intents,document_types
0,train,1367,12,7,10,7
1,test,425,12,7,10,7
2,valid,198,12,7,10,7


In [20]:
def cosine_distance_between_means(left, right):
    left_mean = left.mean(axis=0)
    right_mean = right.mean(axis=0)
    return 1 - float(left_mean @ right_mean) / (
        float(np.linalg.norm(left_mean)) * float(np.linalg.norm(right_mean))
    )


observed_category_distances = []
for category, (group_embeddings, labels, selected_cluster) in cluster_group_data.items():
    selected_mask = labels == selected_cluster
    observed_category_distances.append(
        cosine_distance_between_means(group_embeddings[selected_mask], group_embeddings[~selected_mask])
    )
observed_embedding_shift = float(np.mean(observed_category_distances))

rng = np.random.default_rng(RANDOM_SEED)
permutation_shifts = []
for _ in range(2000):
    category_distances = []
    for group_embeddings, labels, selected_cluster in cluster_group_data.values():
        selected_size = int(np.sum(labels == selected_cluster))
        random_mask = np.zeros(len(group_embeddings), dtype=bool)
        random_mask[rng.choice(len(group_embeddings), size=selected_size, replace=False)] = True
        category_distances.append(
            cosine_distance_between_means(group_embeddings[random_mask], group_embeddings[~random_mask])
        )
    permutation_shifts.append(float(np.mean(category_distances)))

permutation_shifts = np.asarray(permutation_shifts)
random_holdout_shift = float(permutation_shifts.mean())
embedding_shift_ratio = observed_embedding_shift / random_holdout_shift
embedding_shift_pvalue = float(
    (1 + np.sum(permutation_shifts >= observed_embedding_shift)) / (len(permutation_shifts) + 1)
)
assert embedding_shift_pvalue < 0.01

shift_summary = pd.DataFrame(
    [
        {
            "observed_test_shift": observed_embedding_shift,
            "random_holdout_mean": random_holdout_shift,
            "effect_ratio": embedding_shift_ratio,
            "permutation_p_value": embedding_shift_pvalue,
            "permutations": len(permutation_shifts),
        }
    ]
)
display(shift_summary.round(6))

,observed_test_shift,random_holdout_mean,effect_ratio,permutation_p_value,permutations
0,0.037927,0.002932,12.936673,0.0005,2000


In [21]:
split_metric_rows = []
for split, indices in split_indices.items():
    metrics = retrieval_metrics(global_ranks[np.asarray(sorted(indices))])
    split_metric_rows.append({"split": split, **metrics})
split_retrieval_metrics = pd.DataFrame(split_metric_rows).set_index("split").loc[["train", "test", "valid"]].reset_index()
display(split_retrieval_metrics.round(4))

split_distribution = pd.concat(
    [
        df.assign(split=split_membership)
        .groupby(["split", column])
        .size()
        .rename("rows")
        .reset_index()
        .assign(dimension=column)
        .rename(columns={column: "value"})
        for column in stratification_columns
    ],
    ignore_index=True,
)
display(split_distribution)

,split,rows,Recall@1,Recall@5,Recall@10,MRR@10,median_rank
0,train,1367,0.1997,0.4550,0.5647,0.3089,7.0
1,test,425,0.2588,0.5365,0.6659,0.3786,5.0
2,valid,198,0.1818,0.4495,0.5657,0.2931,6.0


,split,value,rows,dimension
0,test,Automotive,30,product_category
1,test,Baby & Toys,34,product_category
2,test,Beauty & Health,19,product_category
3,test,Books & Stationery,34,product_category
4,test,Clothing & Fashion,48,product_category
...,...,...,...,...
103,valid,how_to_guide,29,doc_type
104,valid,product_comparison,32,doc_type
105,valid,product_description,28,doc_type
106,valid,shipping_policy,22,doc_type


## 5. Usefulness assessment

In [22]:
utility_markdown = f"""
### Practical assessment

**Useful for**

- Small-scale contrastive or hard-negative training experiments: each query has one labeled positive and three supplied negatives.
- Cross-lingual query-to-English retrieval experiments across Nepali scripts, Romanized Nepali, mixed queries, and English.
- Testing domain generalization: `test` holds out one complete E5 semantic cluster inside every product category.
- Model diagnostics: the supplied negatives are relatively easy for E5, while retrieval over the full {unique_candidate_documents:,}-document corpus is substantially harder.

**Not yet suitable as**

- A definitive public benchmark or production ground truth without human relevance review.
- Evidence of independent language-mode or document-type performance because those fields are perfectly confounded.
- A clean release without documenting the synthetic generation process, selecting a license, and deciding how to handle the integrity exceptions.

The E5 baseline ranks the positive first against the three provided negatives on **{100 * local_metrics['accuracy']:.1f}%** of valid rows, but reaches only **{100 * global_metrics['Recall@1']:.1f}% Recall@1** and **{100 * global_metrics['Recall@10']:.1f}% Recall@10** over the full corpus. This makes the collection more useful as a synthetic training/evaluation seed than as a mature standalone benchmark.
"""
display(Markdown(utility_markdown))


### Practical assessment

**Useful for**

- Small-scale contrastive or hard-negative training experiments: each query has one labeled positive and three supplied negatives.
- Cross-lingual query-to-English retrieval experiments across Nepali scripts, Romanized Nepali, mixed queries, and English.
- Testing domain generalization: `test` holds out one complete E5 semantic cluster inside every product category.
- Model diagnostics: the supplied negatives are relatively easy for E5, while retrieval over the full 7,959-document corpus is substantially harder.

**Not yet suitable as**

- A definitive public benchmark or production ground truth without human relevance review.
- Evidence of independent language-mode or document-type performance because those fields are perfectly confounded.
- A clean release without documenting the synthetic generation process, selecting a license, and deciding how to handle the integrity exceptions.

The E5 baseline ranks the positive first against the three provided negatives on **91.8%** of valid rows, but reaches only **21.1% Recall@1** and **58.6% Recall@10** over the full corpus. This makes the collection more useful as a synthetic training/evaluation seed than as a mature standalone benchmark.


## 6. Build the Hugging Face DatasetDict and dataset card

In [23]:
hf_dataset = DatasetDict(
    {
        split: Dataset.from_list(
            df.loc[split_membership.eq(split), required_columns].to_dict(orient="records")
        )
        for split in ["train", "test", "valid"]
    }
)
hf_dataset

DatasetDict({
    train: Dataset({
        features: ['row_id', 'document', 'negative1', 'negative2', 'negative3', 'query', 'product_category', 'language_mode', 'query_intent', 'doc_type'],
        num_rows: 1367
    })
    test: Dataset({
        features: ['row_id', 'document', 'negative1', 'negative2', 'negative3', 'query', 'product_category', 'language_mode', 'query_intent', 'doc_type'],
        num_rows: 425
    })
    valid: Dataset({
        features: ['row_id', 'document', 'negative1', 'negative2', 'negative3', 'query', 'product_category', 'language_mode', 'query_intent', 'doc_type'],
        num_rows: 198
    })
})

In [ ]:
def dataframe_to_markdown(table: pd.DataFrame) -> str:
    def format_value(value):
        if pd.isna(value):
            return ""
        if isinstance(value, float):
            return f"{value:g}"
        return str(value).replace("|", "\|").replace("\n", " ")

    columns = [str(column) for column in table.columns]
    header = "| " + " | ".join(columns) + " |"
    separator = "| " + " | ".join("---" for _ in columns) + " |"
    rows = [
        "| " + " | ".join(format_value(value) for value in row) + " |"
        for row in table.itertuples(index=False, name=None)
    ]
    return "\n".join([header, separator, *rows])


composition_overview = pd.DataFrame(
    [
        {
            "dimension": column,
            "values": df[column].nunique(),
            "minimum rows/value": int(df[column].value_counts().min()),
            "maximum rows/value": int(df[column].value_counts().max()),
        }
        for column in dimension_columns
    ]
)

language_card_metrics = metrics_by_dimension["language_mode"][
    ["language_mode", "rows", "Recall@1", "Recall@5", "Recall@10", "MRR@10", "median_rank"]
].copy()
for column in ["Recall@1", "Recall@5", "Recall@10", "MRR@10"]:
    language_card_metrics[column] = language_card_metrics[column].map(lambda value: f"{value:.3f}")

split_card_table = split_coverage.merge(
    split_retrieval_metrics[["split", "Recall@1", "Recall@10", "MRR@10", "median_rank"]],
    on="split",
)
for column in ["Recall@1", "Recall@10", "MRR@10"]:
    split_card_table[column] = split_card_table[column].map(lambda value: f"{value:.3f}")

dataset_info_splits = "\n".join(
    [f"    - name: {split}\n      num_examples: {expected_split_counts[split]}" for split in ["train", "test", "valid"]]
)

card_markdown = f"""---
pretty_name: Nepali E-commerce Retrieval
task_categories:
- text-retrieval
language:
- ne
- en
tags:
- nepali
- ecommerce
- information-retrieval
- cross-lingual
- synthetic
size_categories:
- 1K<n<10K
dataset_info:
  features:
  - name: row_id
    dtype: int64
  - name: document
    dtype: string
  - name: negative1
    dtype: string
  - name: negative2
    dtype: string
  - name: negative3
    dtype: string
  - name: query
    dtype: string
  - name: product_category
    dtype: string
  - name: language_mode
    dtype: string
  - name: query_intent
    dtype: string
  - name: doc_type
    dtype: string
  splits:
{dataset_info_splits}
---

# Nepali E-commerce Retrieval

This is a synthetic e-commerce retrieval dataset containing {len(df):,} query-positive pairs and three supplied negatives per query. Queries cover English, Devanagari Nepali, Romanized Nepali, and mixed modes; the documents are almost entirely English. The most accurate description is therefore **cross-lingual query-to-English retrieval**, not a fully multilingual document corpus.


## Dataset structure

| Field | Description |
|---|---|
| `row_id` | Original integer row identifier; IDs are not contiguous. |
| `query` | E-commerce retrieval query. |
| `document` | Labeled positive document. |
| `negative1`–`negative3` | Supplied negative documents. |
| `product_category` | Product-domain label. |
| `language_mode` | Intended query-language style. |
| `query_intent` | Intended shopping or support intent. |
| `doc_type` | Document-format label. |

The release uses split names exactly `train`, `test`, and `valid`.

{dataframe_to_markdown(split_card_table)}

## Major EDA findings

- **Rows and corpus:** {len(df):,} rows, {len(df.columns)} fields, and {unique_candidate_documents:,} unique candidate documents from {4 * len(df):,} positive/negative slots.
- **Completeness:** no JSON errors, missing fields, nulls, or empty strings were found.
- **Identifier integrity:** `row_id` spans {int(df['row_id'].min())}–{int(df['row_id'].max())} but omits {len(missing_row_ids)} values: `{missing_row_ids}`.
- **Duplicate queries:** {exact_query_duplicate_rows} rows repeat an exact query across {exact_query_duplicate_groups} groups; normalized matching finds {normalized_query_duplicate_rows} duplicate rows. Repeated queries can link to different positive documents.
- **Label collision:** row `{positive_negative_collision_ids[0] if positive_negative_collision_ids else 'none'}` has a positive document duplicated in its negative set.
- **Scripts:** {int((df['query'].map(script_class) == 'Latin only').sum()):,} queries are Latin-only, {int((df['query'].map(script_class) == 'Devanagari only').sum()):,} are Devanagari-only, and {int((df['query'].map(script_class) == 'Devanagari + Latin').sum()):,} mix both. All {len(df):,} positive documents are Latin-only under this check.
- **Confounding:** every `language_mode` maps to exactly one `doc_type` and vice versa. Performance differences between these fields cannot be interpreted independently.
- **Synthetic design:** only {factor_tuple_count:,} of {factor_tuple_theoretical:,} possible category/language/intent/document-type combinations occur.

### Composition

{dataframe_to_markdown(composition_overview)}

### Text lengths

{dataframe_to_markdown(length_summary)}

## E5 retrieval baseline

The baseline uses `{MODEL_ID}` with the required `query:` and `passage:` prefixes and L2-normalized embeddings. Exact normalized duplicate queries treat all associated positive documents as relevant. The invalid four-document row is excluded only from the local four-way metric.

- Four-way positive-at-rank-1 accuracy: **{100 * local_metrics['accuracy']:.2f}%** over {local_metrics['evaluated_rows']:,} valid rows.
- Four-way MRR: **{local_metrics['mrr']:.4f}**.
- Full-corpus Recall@1 / Recall@5 / Recall@10: **{global_metrics['Recall@1']:.4f} / {global_metrics['Recall@5']:.4f} / {global_metrics['Recall@10']:.4f}**.
- Full-corpus MRR@10: **{global_metrics['MRR@10']:.4f}**; median positive rank: **{global_metrics['median_rank']:.0f}**.

### Full-corpus results by labeled language mode

{dataframe_to_markdown(language_card_metrics)}

Because `language_mode` and `doc_type` are one-to-one confounded, this table must not be used to claim a purely linguistic performance difference.

## Split methodology

The split is designed so `valid` remains broadly representative while `test` is a semantic out-of-distribution holdout:

1. Queries are Unicode-NFKC normalized, case-folded, stripped of punctuation, and whitespace-normalized. Each duplicate-query group stays in one split.
2. Each row is represented by the equal-weight concatenation of its normalized E5 query and positive-document embeddings.
3. Query groups are clustered independently inside every product category with five-cluster K-means (`random_state=42`, `n_init=20`).
4. One 10–30%-sized, maximally isolated cluster per category is reserved for `test`. Consequently every product domain occurs in `test`, but its selected E5 clusters do not occur in `train` or `valid`.
5. `valid` targets 10% of all rows and is selected from the remainder by grouped candidate search, minimizing size and marginal-distribution error across category, language mode, intent, and document type. All other rows become `train`.

There is zero normalized-query overlap between splits. Every split contains all {df['product_category'].nunique()} categories, {df['language_mode'].nunique()} language modes, {df['query_intent'].nunique()} intents, and {df['doc_type'].nunique()} document types.

The selected test clusters have mean within-category centroid cosine distance **{observed_embedding_shift:.6f}** from the remainder, compared with **{random_holdout_shift:.6f}** for 2,000 size-matched random holdouts. The shift is **{embedding_shift_ratio:.2f}× larger** (`p={embedding_shift_pvalue:.6f}`, one-sided permutation test), confirming that `test` is materially different in E5 representation space.

## Intended uses

- Contrastive retrieval or reranking experiments using supplied negatives.
- Cross-lingual Nepali/Romanized-Nepali query-to-English retrieval.
- Domain-generalization evaluation with the E5-clustered test split.
- Error analysis across shopping intents and product categories.

## Limitations and release recommendations

- The examples are synthetic/AI-generated and may contain unrealistic details or relevance errors.
- The dataset is small and has no completed human relevance assessment.
- The provided negatives are much easier than retrieval over the full corpus, so four-way accuracy alone overstates retrieval quality.
- Duplicate queries can have multiple linked positives and must be evaluated with multi-positive relevance.
- `language_mode` is inseparable from `doc_type` in the present design.
- Product and price claims should not be treated as current factual information.
- Confirm and add an appropriate license before public upload.

This dataset is best treated as a **synthetic training and evaluation seed**, not as production ground truth or a definitive Nepali retrieval benchmark without cleanup and human validation.
"""

CARD_PATH.write_text(card_markdown, encoding="utf-8")
print(f"Wrote dataset card to {CARD_PATH}")
print(f"Dataset card length: {len(card_markdown):,} characters")

Wrote dataset card to /home/jangedoo/Desktop/projects/nep-lms/nep_lms/datasets/nepali_ecommerce_qa/README.md
Dataset card length: 7,976 characters


<>:7: SyntaxWarning: invalid escape sequence '\|'
<>:7: SyntaxWarning: invalid escape sequence '\|'
/tmp/ipykernel_2043962/3607963424.py:7: SyntaxWarning: invalid escape sequence '\|'
  return str(value).replace("|", "\|").replace("\n", " ")


## 7. Optional Hugging Face upload

The following cell is intentionally disabled. Running all cells will **not** create a repository or upload data. Review the notebook, generated card, split assignments, license, and integrity issues before changing the guard.

In [25]:
if True:  # upload to HF
    from huggingface_hub import HfApi

    hf_dataset.push_to_hub(
        REPO_ID,
        commit_message="Upload semantically split Nepali e-commerce retrieval dataset",
    )
    HfApi().upload_file(
        path_or_fileobj=str(CARD_PATH),
        path_in_repo="README.md",
        repo_id=REPO_ID,
        repo_type="dataset",
        commit_message="Add EDA findings and dataset card",
    )

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

...

New Data Upload: |          |  0.00B /  0.00B            

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            